# Waveform / afterpulse hunt — IC86.22 burnsample

Bruges på den officielle NBI-burnsample fra Niels den Besten:
`/lustre/hpc/project/icecube/Burnsample/I3files/IC86.22/`

1% af IC86.2022 data, processeret som L3 oscNext pass2. `InIceRawData` er
bevaret, så vi kan køre `I3WaveCalibrator` og lave waveform-/afterpulse-studier.

Tidligere version (med janikh's MINIONS sample) er flyttet til [plot_waveforms_MINION.ipynb](plot_waveforms_MINION.ipynb).

In [ ]:
import subprocess
import textwrap
import tempfile
import pickle
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ENV_SHELL = '/cvmfs/icecube.opensciencegrid.org/py3-v4.3.0/RHEL_9_x86_64/metaprojects/icetray/v1.11.1/env-shell.sh'

def run_in_icetray(python_code: str, timeout: int = 7200) -> Path:
    tmp_py  = Path(tempfile.mkstemp(suffix='.py')[1])
    tmp_pkl = Path(tempfile.mkstemp(suffix='.pkl')[1])
    tmp_py.write_text(textwrap.dedent(python_code))
    proc = subprocess.Popen(
        [ENV_SHELL, 'python', '-u', str(tmp_py)],
        env={**os.environ, 'OUT_PICKLE': str(tmp_pkl), 'PYTHONUNBUFFERED': '1'},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line.rstrip())
            sys.stdout.flush()
        proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        raise
    if proc.returncode != 0:
        raise RuntimeError(f'icetray subprocess failed (rc={proc.returncode})')
    return tmp_pkl

## Scan-konfiguration

**Datakilde:** L3 oscNext pass2 burnsample (1% af real data, IC86.2022).
Hver run har sin egen GCD i `GCD/`-undermappen — vi skal matche dem.

Som default kører vi over første subrun-fil for at gå hurtigt. Læg flere
filer i `I3_FILES` listen hvis du vil scanne bredere.

In [ ]:
import re
BURN_ROOT = Path('/lustre/hpc/project/icecube/Burnsample/I3files/IC86.22')
GCD_DIR   = BURN_ROOT / 'GCD'

# Glob across many runs (each with its own GCD) to get enough statistics.
# Each run typically has 3-4 subruns × ~1000 events.
N_RUNS = 1366   # ~40 runs × 3 subruns × 1000 evts ≈ 120k events scanned

def _build_file_list(n_runs):
    """Build [GCD_run1, sub_run1a, sub_run1b, ..., GCD_run2, sub_run2a, ...]
    so I3Reader injects the correct calibration per run."""
    runs = sorted({re.search(r'Run(\d{8})', p.name).group(1)
                   for p in BURN_ROOT.glob('oscNext_data_*.i3.zst')})
    files = []
    kept = 0
    for run in runs:
        if kept >= n_runs:
            break
        gcd = next(iter(GCD_DIR.glob(f'Level2_IC86.2022_data_Run{run}_*_GCD.i3.zst')), None)
        if gcd is None:
            continue
        subruns = sorted(BURN_ROOT.glob(
            f'oscNext_data_IC86.22_*_Run{run}_Subrun*.i3.zst'))
        if not subruns:
            continue
        files.append(str(gcd))
        files.extend(str(p) for p in subruns)
        kept += 1
    return files, kept

FILE_LIST, n_runs_kept = _build_file_list(N_RUNS)
print(f'using {n_runs_kept} runs, {len(FILE_LIST)} files total (incl. per-run GCDs)')

# Afterpulse-search criteria
N_FRAMES         = 500000        # cap, won't actually hit
PROGRESS_EVERY   = 5000
MAIN_WINDOW_NS   = 300.0
MAIN_PEAK_MIN    = 20
AP_WINDOW_NS     = (2000.0, 10000.0)
AP_PEAK_MIN      = 8
CONTRAST_MIN     = 8
EARLY_STOP_PEAK  = 12.0
EARLY_STOP_CTR   = 15.0

In [ ]:
files_repr = repr(FILE_LIST)
scan_code = f'''
    import os, math, pickle, statistics, sys, time
    from icecube import icetray, dataio, dataclasses, WaveCalibrator
    from icecube.icetray import I3Tray, I3Units

    icetray.logging.set_level_for_unit("I3WaveCalibrator", "FATAL")
    icetray.logging.set_level_for_unit("I3Reader", "WARN")

    FE_R, E_CHG = 50.0, 1.602176634e-19
    MAIN_W       = {MAIN_WINDOW_NS}
    MAIN_PEAK_MIN= {MAIN_PEAK_MIN}
    AP_LO, AP_HI = {AP_WINDOW_NS[0]}, {AP_WINDOW_NS[1]}
    AP_PEAK_MIN  = {AP_PEAK_MIN}
    CONTRAST_MIN = {CONTRAST_MIN}
    EARLY_STOP_PEAK = {EARLY_STOP_PEAK}
    EARLY_STOP_CTR  = {EARLY_STOP_CTR}
    PROGRESS_EVERY  = {PROGRESS_EVERY}
    N_FRAMES        = {N_FRAMES}

    best = {{"score": -1.0}}
    candidates = []
    found = [False]
    seen  = 0
    t_start = time.time()

    def scan(frame):
        global seen, best
        seen += 1
        if seen % PROGRESS_EVERY == 0:
            elapsed = time.time() - t_start
            rate = seen / elapsed if elapsed > 0 else 0
            cur = best.get("ap_peak_pe", 0.0)
            print(f"  {{seen:>7d}} / {{N_FRAMES}} checked  "
                  f"({{elapsed:5.0f}}s, {{rate:5.0f}} frames/s)  "
                  f"best AP peak so far: {{cur:.1f}} PE  cands: {{len(candidates)}}",
                  flush=True)
        if seen > N_FRAMES or found[0]: return
        if "InIceRawData" not in frame or "CalibratedWaveforms" not in frame: return
        rd, cal_wfs = frame["InIceRawData"], frame["CalibratedWaveforms"]
        cal, det = frame["I3Calibration"], frame["I3DetectorStatus"]
        for om, launches in rd:
            L = next((x for x in launches if x.lc_bit), None)
            if L is None or om not in cal_wfs: continue
            if om not in cal.dom_cal or om not in det.dom_status: continue
            dc, ds = cal.dom_cal[om], det.dom_status[om]
            hv = float(ds.pmt_hv) / I3Units.V
            if hv <= 0: continue
            gain = 10 ** (dc.hv_gain_fit.intercept + dc.hv_gain_fit.slope * math.log10(hv))
            kf = 1.0 / (FE_R * gain * E_CHG)
            fadc = next((w for w in cal_wfs[om] if str(w.source) == "FADC"), None)
            if fadc is None: continue
            V = [float(v) / I3Units.V for v in fadc.waveform]
            dt_s = float(fadc.bin_width) * 1e-9
            pe = [v * dt_s * kf for v in V]
            t0 = float(fadc.time)
            dt_ns = float(fadc.bin_width)
            t_main = float(L.time)

            main_peak = max(
                (q for i, q in enumerate(pe)
                 if abs((t0 + i * dt_ns) - t_main) <= MAIN_W),
                default=0.0,
            )
            if main_peak < MAIN_PEAK_MIN: continue

            ap_vals = []
            ap_peak = 0.0
            ap_peak_t = None
            for i, q in enumerate(pe):
                ti = t0 + i * dt_ns
                if AP_LO <= (ti - t_main) <= AP_HI:
                    ap_vals.append(q)
                    if q > ap_peak:
                        ap_peak = q; ap_peak_t = ti
            if len(ap_vals) < 20 or ap_peak < AP_PEAK_MIN: continue

            med = statistics.median(ap_vals)
            base = max(med, 0.05)
            contrast = ap_peak / base
            if contrast < CONTRAST_MIN: continue

            score = contrast * ap_peak
            if score > best["score"]:
                hdr = frame["I3EventHeader"]
                payload = {{
                    "score": score,
                    "frame_no": seen,
                    "ap_peak_pe": ap_peak,
                    "ap_peak_t_ns": ap_peak_t,
                    "ap_median_pe": med,
                    "ap_contrast": contrast,
                    "ap_sum_pe": sum(ap_vals),
                    "q_fadc_pe": sum(pe),
                    "main_peak_pe": main_peak,
                    "om": (int(om.string), int(om.om), int(om.pmt)),
                    "pmt_gain": float(gain),
                    "pmt_hv_volts": hv,
                    "pe_per_voltsecond": kf,
                    "hlc_launch_time_ns": t_main,
                    "event": {{"run_id": hdr.run_id, "event_id": hdr.event_id}},
                    "calibrated_waveforms": [
                        {{"source": str(w.source), "channel": int(w.channel),
                          "time_ns": float(w.time), "bin_width_ns": float(w.bin_width),
                          "samples_volt": [float(v) / I3Units.V for v in w.waveform]}}
                        for w in cal_wfs[om]
                    ],
                }}
                best = payload
                candidates.append(payload)
                print(f"    -> new best at frame {{seen}}: main peak = {{main_peak:.1f}} PE, "
                      f"AP peak = {{ap_peak:.2f}} PE, contrast = {{contrast:.1f}}x  "
                      f"[run {{hdr.run_id}} event {{hdr.event_id}} DOM {{(int(om.string), int(om.om), int(om.pmt))}}]", flush=True)
                if ap_peak > EARLY_STOP_PEAK and contrast > EARLY_STOP_CTR:
                    found[0] = True; return

    def drop_existing(frame):
        for k in ("CalibratedWaveforms", "CalibrationErrata"):
            if k in frame: del frame[k]

    def has_launches(frame):
        return "InIceRawData" in frame

    def stop(frame):
        if seen >= N_FRAMES or found[0]: tray.RequestSuspension()

    tray = I3Tray()
    tray.Add("I3Reader", FilenameList={files_repr})
    tray.Add(drop_existing, Streams=[icetray.I3Frame.DAQ])
    tray.Add("I3WaveCalibrator",
             If=has_launches,
             Launches="InIceRawData",
             Waveforms="CalibratedWaveforms",
             WaveformRange="CalibratedWaveformRange_recalc")
    tray.Add(scan, Streams=[icetray.I3Frame.DAQ])
    tray.Add(stop, Streams=[icetray.I3Frame.DAQ])
    tray.Execute()

    with open(os.environ["OUT_PICKLE"], "wb") as f:
        pickle.dump({{"best": best, "candidates": candidates}},
                    f, protocol=pickle.HIGHEST_PROTOCOL)

    print()
    print("=" * 70)
    print(f"final: {{len(candidates)}} candidates from {{seen}} frames")
    print(f"  {{'frame':>7s}}  {{'run':>7s}}  {{'event':>10s}}  {{'DOM':>15s}}  "
          f"{{'main':>6s}}  {{'AP':>6s}}  {{'ctr':>7s}}")
    for c in candidates:
        print(f"  {{c['frame_no']:>7d}}  {{c['event']['run_id']:>7d}}  "
              f"{{c['event']['event_id']:>10d}}  {{str(c['om']):>15s}}  "
              f"{{c['main_peak_pe']:>6.1f}}  {{c['ap_peak_pe']:>6.2f}}  "
              f"{{c['ap_contrast']:>6.1f}}x")
'''

pkl_path = run_in_icetray(scan_code)
with open(pkl_path, 'rb') as f:
    result = pickle.load(f)
best = result['best']
candidates = result['candidates']
print(f"\n→ {len(candidates)} candidates kept in `candidates` list.")

## Pick a specific candidate to plot

Sæt `PICK_INDEX` til en index i `candidates`-listen, eller giv direkte
`TARGET_RUN`/`TARGET_EVENT`/`TARGET_OM`. Hvis target ikke findes i candidates
åbner notebooken I3-filen igen for at hente waveform'en præcis.

In [ ]:
TARGET_RUN   = None
TARGET_EVENT = None
TARGET_OM    = None
PICK_INDEX   = None

if PICK_INDEX is not None:
    best = candidates[PICK_INDEX]
    print(f"Loaded candidates[{PICK_INDEX}]: "
          f"run {best['event']['run_id']} event {best['event']['event_id']} DOM {best['om']}")
elif TARGET_RUN is not None and TARGET_EVENT is not None:
    hit = next(
        (c for c in candidates
         if c['event']['run_id'] == TARGET_RUN
         and c['event']['event_id'] == TARGET_EVENT
         and (TARGET_OM is None or c['om'] == TARGET_OM)),
        None,
    )
    if hit is not None:
        best = hit
        print(f"Loaded from candidates: run {TARGET_RUN} event {TARGET_EVENT} DOM {best['om']}")
    else:
        # Re-fetch from I3 — only need the specific run's files + its GCD
        gcd_match = next(iter(GCD_DIR.glob(
            f'Level2_IC86.2022_data_Run{TARGET_RUN:08d}_*_GCD.i3.zst')), None)
        if gcd_match is None:
            raise FileNotFoundError(f'no GCD for run {TARGET_RUN}')
        sub_files = sorted(BURN_ROOT.glob(
            f'oscNext_data_IC86.22_*_Run{TARGET_RUN:08d}_Subrun*.i3.zst'))
        fetch_files = [str(gcd_match)] + [str(p) for p in sub_files]
        files_repr = repr(fetch_files)
        print(f"Not in candidates — fetching from {len(sub_files)} subrun files of run {TARGET_RUN} ...")
        fetch_code = f'''
            import os, math, pickle
            from icecube import icetray, dataio, dataclasses, WaveCalibrator
            from icecube.icetray import I3Tray, I3Units

            icetray.logging.set_level_for_unit("I3WaveCalibrator", "FATAL")

            FE_R, E_CHG = 50.0, 1.602176634e-19
            TARGET_RUN, TARGET_EVENT = {TARGET_RUN}, {TARGET_EVENT}
            TARGET_OM = {TARGET_OM!r}
            out = [None]

            def grab(frame):
                if out[0]: return
                if "InIceRawData" not in frame or "CalibratedWaveforms" not in frame: return
                hdr = frame["I3EventHeader"]
                if hdr.run_id != TARGET_RUN or hdr.event_id != TARGET_EVENT: return
                rd, cal_wfs = frame["InIceRawData"], frame["CalibratedWaveforms"]
                cal, det = frame["I3Calibration"], frame["I3DetectorStatus"]
                for om, launches in rd:
                    om_t = (int(om.string), int(om.om), int(om.pmt))
                    if TARGET_OM is not None and om_t != TARGET_OM: continue
                    L = next((x for x in launches if x.lc_bit), None)
                    if L is None or om not in cal_wfs: continue
                    if om not in cal.dom_cal or om not in det.dom_status: continue
                    dc, ds = cal.dom_cal[om], det.dom_status[om]
                    hv = float(ds.pmt_hv) / I3Units.V
                    gain = 10 ** (dc.hv_gain_fit.intercept + dc.hv_gain_fit.slope * math.log10(hv))
                    kf = 1.0 / (FE_R * gain * E_CHG)
                    out[0] = {{
                        "om": om_t,
                        "pmt_gain": float(gain),
                        "pmt_hv_volts": hv,
                        "pe_per_voltsecond": kf,
                        "hlc_launch_time_ns": float(L.time),
                        "event": {{"run_id": hdr.run_id, "event_id": hdr.event_id}},
                        "calibrated_waveforms": [
                            {{"source": str(w.source), "channel": int(w.channel),
                              "time_ns": float(w.time), "bin_width_ns": float(w.bin_width),
                              "samples_volt": [float(v) / I3Units.V for v in w.waveform]}}
                            for w in cal_wfs[om]
                        ],
                    }}
                    break

            def drop_existing(frame):
                for k in ("CalibratedWaveforms", "CalibrationErrata"):
                    if k in frame: del frame[k]

            def has_launches(frame):
                return "InIceRawData" in frame

            def stop(frame):
                if out[0]: tray.RequestSuspension()

            tray = I3Tray()
            tray.Add("I3Reader", FilenameList={files_repr})
            tray.Add(drop_existing, Streams=[icetray.I3Frame.DAQ])
            tray.Add("I3WaveCalibrator",
                     If=has_launches,
                     Launches="InIceRawData",
                     Waveforms="CalibratedWaveforms",
                     WaveformRange="CalibratedWaveformRange_recalc")
            tray.Add(grab, Streams=[icetray.I3Frame.DAQ])
            tray.Add(stop, Streams=[icetray.I3Frame.DAQ])
            tray.Execute()

            if out[0] is None:
                raise RuntimeError("event/DOM not found")
            with open(os.environ["OUT_PICKLE"], "wb") as f:
                pickle.dump(out[0], f, protocol=pickle.HIGHEST_PROTOCOL)
            ev = out[0]["event"]
            print(f"fetched DOM {{out[0]['om']}}  run={{ev['run_id']}} event={{ev['event_id']}}")
        '''
        fp = run_in_icetray(fetch_code)
        with open(fp, 'rb') as f:
            best = pickle.load(f)
else:
    print("No target set — `best` is unchanged.")

## Plot the chosen waveform

In [ ]:
k = best['pe_per_voltsecond']
t_main = best['hlc_launch_time_ns']

# Group waveforms by source; for ATWD keep all 3 channels (ch0/1/2).
atwd_wfs = [w for w in best['calibrated_waveforms'] if w['source'] == 'ATWD']
fadc_wfs = [w for w in best['calibrated_waveforms'] if w['source'] == 'FADC']

# If multiple launches, restrict to the launch closest to t_main
def near_t_main(wfs):
    if not wfs: return []
    launch_t = min((w['time_ns'] for w in wfs), key=lambda t: abs(t - t_main))
    return [w for w in wfs if abs(w['time_ns'] - launch_t) < 1.0]

atwd_wfs = sorted(near_t_main(atwd_wfs), key=lambda w: w['channel'])
fadc_wfs = near_t_main(fadc_wfs)

fig, axes = plt.subplots(2, 1, figsize=(11, 6.5))

# --- ATWD: overlay all channels ---
ax = axes[0]
ch_colors = {0: 'C0', 1: 'C2', 2: 'C3'}
for w in atwd_wfs:
    V = np.asarray(w['samples_volt'])
    dt_s = w['bin_width_ns'] * 1e-9
    pe = V * dt_s * k
    t = w['time_ns'] + np.arange(len(pe)) * w['bin_width_ns']
    ax.step(t, pe, where='post', lw=1.2,
            color=ch_colors.get(w['channel'], 'gray'),
            label=f"ATWD ch{w['channel']}  (total {pe.sum():.1f} PE)")
ax.axvline(t_main, color='red', ls=':', lw=1, alpha=0.7,
           label=f't_main = {t_main:.0f} ns')
ax.set_xlabel('time [ns]')
ax.set_ylabel('charge per sample [PE]')
ax.set_title(f"ATWD — 3 gain channels (ch0 saturates first, ch2 last)")
ax.grid(alpha=0.3)
ax.legend(loc='upper right', fontsize=8)

# --- FADC ---
ax = axes[1]
if fadc_wfs:
    w = fadc_wfs[0]
    V = np.asarray(w['samples_volt'])
    dt_s = w['bin_width_ns'] * 1e-9
    pe = V * dt_s * k
    t = w['time_ns'] + np.arange(len(pe)) * w['bin_width_ns']
    ax.step(t, pe, where='post', lw=1.2, color='C1',
            label=f"FADC ch{w['channel']}  (total {pe.sum():.1f} PE)")
    ax.axvline(t_main, color='red', ls=':', lw=1, alpha=0.7,
               label=f't_main = {t_main:.0f} ns')
    ax.axvspan(t_main + 300, t_main + 12000, color='orange', alpha=0.12,
               label='afterpulse window (0.3-12 µs)')
    ax.set_title(f"FADC  ({len(pe)} samples × {w['bin_width_ns']:.2f} ns/bin)")
    ax.legend(loc='upper right', fontsize=8)
ax.set_xlabel('time [ns]')
ax.set_ylabel('charge per sample [PE]')
ax.grid(alpha=0.3)

fig.suptitle(
    f"IC86.22 burnsample — run {best['event']['run_id']} event {best['event']['event_id']} DOM {best['om']}",
    fontsize=12, y=1.02,
)
fig.tight_layout()
plt.show()